# Behavior Discovery Method Comparison Matrix

Objective: keep dataset x feature x discovery comparisons explicit, reproducible, and easy to extend. The default run is lightweight; optional methods become active when their dependencies are installed.

## Method Families And Assumptions

This matrix compares method families rather than just packages:

- Geometry-first clustering: PCA/UMAP/KMeans gives a quick baseline for whether the feature space already separates states.
- Temporal segmentation: B-SOiD and keypoint-MoSeq add time structure, so the labels are syllables or states rather than independent cluster IDs.
- Spectral hierarchy: SUBTLE turns motion into a time-frequency representation and then discovers hierarchical motifs.
- Pretrained embeddings: hBehaveMAE and similar encoders reuse a learned latent space and cluster that space instead of raw pose.

The comparison is only meaningful if you keep feature block, temporal scale, and geometry fixed while changing one discovery family at a time.

### Where CEBRA Fits

Treat CEBRA as a feature / embedding generator. If the input is behavior-only, use temporal windows or weak behavioral context to define positive pairs. If the input also includes neural data, the same framework can align the two modalities.

That is why CEBRA belongs in the feature layer of the matrix, not the discovery layer. The downstream clustering or syllable extraction still needs its own method and its own temporal metrics.


In [1]:
from pathlib import Path
import sys
import importlib.util
import numpy as np
try:
    import pandas as pd
except ImportError:
    pd = None

ROOT = Path.cwd()
if ROOT.name != "behavior-lab":
    ROOT = Path("/Users/joon/dev/behavior-lab")
sys.path.insert(0, str(ROOT / "src"))

from behavior_lab.data.loaders import get_loader
from behavior_lab.data.features import list_feature_blocks, list_discovery_methods
from behavior_lab.experiments import compare_discovery_methods

RANDOM_STATE = 42
MAX_FRAMES = 900
DATA_ROOT = ROOT / "data"

def show_table(rows):
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)

## Comparison Design

In [2]:
feature_options = [s.name for s in list_feature_blocks()]
method_options = [s.name for s in list_discovery_methods()]
rows = []
for i in range(max(len(feature_options), len(method_options))):
    rows.append({
        "feature_blocks": feature_options[i] if i < len(feature_options) else "",
        "discovery_methods": method_options[i] if i < len(method_options) else "",
    })
show_table(rows)

,feature_blocks,discovery_methods
0,raw_keypoints,kmeans_pca_umap
1,skeleton_kinematic,B-SOiD
2,dyadic_egocentric,keypoint-moseq
3,bsoid_spatiotemporal,pca_hmm_fallback
4,morlet_cwt,SUBTLE
5,self_supervised_embedding,hBehaveMAE


## Dataset Loader

In [3]:
def load_dataset(name="calms21"):
    if name == "calms21" and (DATA_ROOT / "calms21").exists():
        return get_loader("calms21", data_dir=DATA_ROOT / "calms21").load_split("train")[0]
    if name == "mabe22" and (DATA_ROOT / "mabe22").exists():
        return get_loader("mabe22", data_dir=DATA_ROOT / "mabe22").load_split("train")[0]
    rng = np.random.default_rng(RANDOM_STATE)
    kp = rng.normal(size=(1200, 6, 2)).astype("float32").cumsum(axis=0)
    from behavior_lab.core.types import BehaviorSequence
    return BehaviorSequence(keypoints=kp, fps=30, sample_id="synthetic_walk")

seq = load_dataset("calms21")
keypoints = np.nan_to_num(seq.keypoints[:MAX_FRAMES].astype("float32"))
print(seq.sample_id, keypoints.shape, seq.fps)

train_00000 (64, 14, 2) 30.0


## Method Availability

`kmeans_pca_umap` is the default baseline. Enable heavier methods after installing extras: `behavior-lab[bsoid]`, `behavior-lab[moseq]`, `behavior-lab[moseq-fallback]`, `behavior-lab[subtle]`, or the relevant torch/checkpoint setup for hBehaveMAE.

## What To Compare

For each run, record:

- silhouette for geometric separation
- ARI/NMI only when labels are actually comparable
- number of bouts or states
- mean bout duration
- runtime and failure mode

A method with a weaker silhouette can still be preferable if it produces a more stable motif grammar or a more interpretable transition structure.

In [4]:
availability = {
    "kmeans_pca_umap": True,
    "bsoid": importlib.util.find_spec("hdbscan") is not None and importlib.util.find_spec("umap") is not None,
    "pca_hmm_fallback": importlib.util.find_spec("hmmlearn") is not None,
    "moseq": importlib.util.find_spec("keypoint_moseq") is not None,
    "subtle": importlib.util.find_spec("subtle") is not None,
    "behavemae": importlib.util.find_spec("torch") is not None,
}
availability

{'kmeans_pca_umap': True,
 'bsoid': True,
 'pca_hmm_fallback': True,
 'moseq': True,
 'subtle': True,
 'behavemae': True}

## Run A Controlled Sweep

In [5]:
selected_methods = ["kmeans_pca_umap"]
if availability["bsoid"]:
    selected_methods.append("bsoid")
if availability["pca_hmm_fallback"]:
    selected_methods.append("pca_hmm_fallback")

runs = compare_discovery_methods(
    keypoints,
    methods=selected_methods,
    feature="skeleton_kinematic",
    fps=seq.fps,
    n_clusters=8,
    random_state=RANDOM_STATE,
)

summary = []
for run in runs:
    bm = run.behavior_metrics
    summary.append({
        "dataset": seq.sample_id,
        "method": run.name,
        "feature": run.feature_name,
        "frames_or_bins": run.result.num_frames,
        "clusters": run.result.n_clusters,
        "bouts": bm.num_bouts if bm else None,
        "silhouette": (run.cluster_metrics or {}).get("silhouette"),
        "notes": (run.cluster_metrics or {}).get("error", ""),
    })
show_table(summary)

/opt/anaconda3/envs/animal_behavior/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/opt/anaconda3/envs/animal_behavior/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/opt/anaconda3/envs/animal_behavior/lib/python3.10/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


,dataset,method,feature,frames_or_bins,clusters,bouts,silhouette,notes
0,train_00000,kmeans_pca_umap,skeleton_kinematic,64,8,27,0.042788,
1,train_00000,bsoid,skeleton_kinematic,21,0,1,0.000000,
2,train_00000,pca_hmm_fallback,skeleton_kinematic,64,8,35,0.141915,


## Interpretation Checklist

- Pose layer: keypoint source, confidence handling, triangulation quality, coordinate units.
- Feature layer: raw geometry vs kinematic summaries vs spectral/learned embeddings.
- Discovery layer: fixed cluster count, density clusters, temporal HMM/SLDS syllables, hierarchy.
- Dynamics layer: bout duration distribution and transition matrix, not only embedding separation.
- Dataset layer: species, number of animals, 2D/3D geometry, labels available or not.